# TARDIS — Step 2 : Data Visualization & Analysis

Ce notebook part de `cleaned_dataset.csv` produit au Step 1.
Il calcule des statistiques descriptives, trace des graphiques
et donne une interpretation ecrite apres chaque resultat.

## 1. Chargement des donnees nettoyees

In [1]:
# os : verification de l existence du fichier selon le dossier d execution
import os

# matplotlib.pyplot : bibliotheque de base pour tracer des graphiques
import matplotlib.pyplot as plt

# pandas : manipulation du tableau de donnees
import pandas as pd

# seaborn : bibliotheque de graphiques construite au dessus de matplotlib
import seaborn as sns

In [2]:
# pd.read_csv() lit le fichier produit par le Step 1
# parse_dates=["date"] demande a pandas de relire la colonne date comme une vraie date
chemin = "cleaned_dataset.csv"
if not os.path.exists(chemin) and os.path.exists("../cleaned_dataset.csv"):
    chemin = "../cleaned_dataset.csv"

df = pd.read_csv(chemin, parse_dates=["date"])

# .shape donne (nombre de lignes, nombre de colonnes)
print("Taille du dataset :", df.shape)

Taille du dataset : (11278, 36)


In [3]:
# sns.set_theme() applique un style visuel a tous les graphiques du notebook
# style="whitegrid" : fond blanc avec une grille legere, plus lisible
sns.set_theme(style="whitegrid")

# plt.rcParams définit les reglages par defaut de matplotlib
# figure.figsize : largeur et hauteur des graphiques en pouces
plt.rcParams["figure.figsize"] = (9, 5)

# On fixe une couleur principale, utilisee dans tous les graphiques a une seule serie.
# Garder une seule couleur evite les graphiques multicolores illisibles.
COULEUR = "#2563eb"

## 2. Statistiques descriptives

In [4]:
# On choisit les colonnes les plus importantes a decrire
colonnes_cles = [
    "avg_delay_all_arr",
    "avg_delay_late_arr",
    "delay_rate_arr",
    "punctuality_rate",
    "cancellation_rate",
    "journey_time",
    "nb_scheduled",
]

# .describe() calcule d'un coup : le nombre de valeurs, la moyenne, l'ecart-type,
# le minimum, les quartiles (25%, 50%, 75%) et le maximum
# .T (transpose) fait pivoter le tableau pour avoir une ligne par variable
# .round(2) arrondit a 2 decimales
df[colonnes_cles].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
avg_delay_all_arr,11278.0,6.22,4.21,-15.19,3.44,5.39,8.13,92.00
avg_delay_late_arr,11278.0,35.42,15.35,0.00,26.09,33.45,42.33,299.60
delay_rate_arr,11278.0,13.99,8.03,0.00,8.75,12.61,17.50,100.00
punctuality_rate,11278.0,86.01,8.03,0.00,82.50,87.39,91.25,100.00
cancellation_rate,11278.0,3.85,8.64,0.00,0.00,0.71,3.03,92.68
journey_time,11278.0,171.89,86.65,0.00,101.00,164.00,222.00,703.13
nb_scheduled,11278.0,271.95,179.78,1.00,154.00,229.00,356.00,1100.00


Lecture du tableau :

- le retard moyen a l'arrivee est de **6,2 minutes**, la mediane de **5,4 minutes**
- la moyenne est superieure a la mediane, donc la distribution est etiree
  vers la droite : quelques trajets tres en retard tirent la moyenne vers le haut
- le maximum atteint **92 minutes** de retard moyen sur un mois
- le taux de ponctualite moyen est d'environ **86 %**
- le taux d'annulation moyen est de **3,9 %** mais sa mediane n'est que de **0,7 %** :
  quelques mois de greve font monter la moyenne


In [5]:
# .value_counts() compte le nombre de lignes de chaque categorie
# normalize=True donne une proportion au lieu d'un effectif
# * 100 pour avoir un pourcentage, .round(1) pour arrondir
repartition = df["delay_category"].value_counts(normalize=True) * 100
print(repartition.round(1))

delay_category
Faible       45.2
Modere       40.1
Important    13.6
Critique      1.1
Name: proportion, dtype: float64


In [6]:
# .groupby("service") regroupe les lignes par type de service
# ["avg_delay_all_arr"] choisit la colonne a resumer
# .agg(["mean", "median", "count"]) calcule plusieurs statistiques d'un coup
df.groupby("service")["avg_delay_all_arr"].agg(["mean", "median", "count"]).round(2)

,mean,median,count
service,,,
INTERNATIONAL,8.12,7.18,1349
NATIONAL,5.97,5.21,9929


In [7]:
# Meme principe, mais regroupe par gare de depart
# .sort_values("mean", ascending=False) trie du plus grand retard au plus petit
stats_gares = df.groupby("departure_station")["avg_delay_all_arr"].agg(
    ["mean", "count"]
)
stats_gares = stats_gares.sort_values("mean", ascending=False).round(2)

# .head(10) affiche les 10 premieres lignes, donc les 10 gares les plus en retard
stats_gares.head(10)

,mean,count
departure_station,,
ITALIE,14.38,88
MADRID,12.14,35
BARCELONA,10.44,91
CHAMBERY CHALLES LES EAUX,10.31,95
TOULOUSE MATABIAU,10.08,93
MACON LOCHE,9.07,91
PERPIGNAN,8.94,92
MARNE LA VALLEE,8.93,175
STUTTGART,8.92,90


Les statistiques par gare montrent deja un ecart important : les gares en tete
depassent 10 minutes de retard moyen alors que les mieux placees restent sous
4 minutes. La gare de depart est donc une variable utile pour le modele du Step 3.

## 3. Distribution des retards

In [8]:
# plt.hist() trace un histogramme : il decoupe les valeurs en tranches
# et compte combien de trajets tombent dans chaque tranche
# bins=50 : nombre de tranches
# color : couleur des barres, edgecolor : couleur du contour
plt.hist(df["avg_delay_all_arr"], bins=50, color=COULEUR, edgecolor="white")

# plt.title() met un titre au graphique
plt.title("Distribution du retard moyen a l'arrivee")

# plt.xlabel() et plt.ylabel() nomment les axes
plt.xlabel("Retard moyen (minutes)")
plt.ylabel("Nombre de trajets")

# plt.show() affiche le graphique
plt.show()

/tmp/ipykernel_5920/3797865235.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


La distribution est asymetrique : la grande majorite des trajets se situe entre
2 et 10 minutes de retard, avec un pic autour de 5 minutes. La queue a droite est
longue mais peu peuplee, ce sont des mois exceptionnels (greve, incident majeur).

Consequence pour le Step 3 : la variable a predire n'est pas symetrique, il faudra
regarder l'erreur absolue moyenne (MAE) autant que le RMSE, car le RMSE est
fortement penalise par ces valeurs extremes.

In [9]:
# sns.boxplot() trace une boite a moustaches : elle resume la distribution
# la boite va du 1er au 3e quartile, le trait au milieu est la mediane,
# les points a l'exterieur sont les valeurs extremes
sns.boxplot(x=df["avg_delay_all_arr"], color=COULEUR)

plt.title("Boite a moustaches du retard moyen a l'arrivee")
plt.xlabel("Retard moyen (minutes)")
plt.show()

/tmp/ipykernel_5920/1969729175.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# .value_counts() compte les trajets de chaque categorie de retard
# .sort_index() range les categories dans l'ordre Faible, Modere, Important, Critique
comptage = df["delay_category"].value_counts().sort_index()

# plt.bar() trace un diagramme en barres verticales
# comptage.index : les noms des categories, comptage.values : les effectifs
plt.bar(comptage.index, comptage.values, color=COULEUR)

plt.title("Nombre de trajets par categorie de retard")
plt.xlabel("Categorie de retard")
plt.ylabel("Nombre de trajets")
plt.show()

/tmp/ipykernel_5920/869700826.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Environ 85 % des trajets sont en retard faible ou modere (moins de 10 minutes).
Les situations critiques (plus de 20 minutes) representent a peine 1 % des cas.
Le jeu de donnees est donc desequilibre : un modele qui predirait toujours
6 minutes se tromperait rarement de beaucoup, ce qui explique l'exigence du sujet
de battre une baseline.

## 4. Comparaison entre gares

In [11]:
# On calcule le retard moyen par gare de depart
retard_par_gare = df.groupby("departure_station")["avg_delay_all_arr"].mean()

# .sort_values() trie les valeurs
# .tail(12) prend les 12 dernieres, donc les plus elevees
top_gares = retard_par_gare.sort_values().tail(12)

# .plot(kind="barh") trace des barres horizontales
# Les barres horizontales permettent de lire les noms de gares sans les incliner
top_gares.plot(kind="barh", color=COULEUR)

plt.title("12 gares de depart avec le retard moyen le plus eleve")
plt.xlabel("Retard moyen a l'arrivee (minutes)")
plt.ylabel("Gare de depart")
plt.show()

/tmp/ipykernel_5920/4275757391.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Meme graphique pour les gares les plus ponctuelles
# .head(12) prend les 12 premieres du tri croissant, donc les plus faibles retards
meilleures_gares = retard_par_gare.sort_values().head(12)

meilleures_gares.plot(kind="barh", color=COULEUR)

plt.title("12 gares de depart les plus ponctuelles")
plt.xlabel("Retard moyen a l'arrivee (minutes)")
plt.ylabel("Gare de depart")
plt.show()

/tmp/ipykernel_5920/708628144.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


L'ecart entre les deux graphiques est important : Italie (14,4 minutes),
Madrid (12,1) et Barcelone (10,4) sont en tete, alors que Reims (2,8),
Nancy (3,4) et Le Creusot (3,6) restent sous 4 minutes.

Deux facteurs se cumulent : les longues distances et les liaisons transfrontalieres,
qui accumulent les aleas sur le parcours. La gare de depart est confirmee comme
variable explicative importante.

## 5. Effet du temps

In [13]:
# On regroupe par mois et on calcule le retard moyen de chaque mois
retard_par_mois = df.groupby("month")["avg_delay_all_arr"].mean()

# .plot(marker="o") trace une courbe avec un point sur chaque valeur
plt.plot(retard_par_mois.index, retard_par_mois.values, marker="o", color=COULEUR)

plt.title("Retard moyen selon le mois de l'annee")
plt.xlabel("Mois")
plt.ylabel("Retard moyen (minutes)")

# plt.xticks() force l'affichage des 12 numeros de mois sur l'axe horizontal
plt.xticks(range(1, 13))
plt.show()

/tmp/ipykernel_5920/3294954130.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Le retard suit un cycle annuel net : creux en mai (5,0 minutes), pic en juillet
(8,4 minutes), remontee secondaire en novembre. Juillet cumule le trafic de
vacances et les travaux d'ete sur le reseau.

Le mois est donc une variable temporelle utile pour la prediction.

In [14]:
# On regroupe par annee
retard_par_annee = df.groupby("year")["avg_delay_all_arr"].mean()

plt.plot(retard_par_annee.index, retard_par_annee.values, marker="o", color=COULEUR)

plt.title("Evolution du retard moyen par annee")
plt.xlabel("Annee")
plt.ylabel("Retard moyen (minutes)")
plt.show()

/tmp/ipykernel_5920/2341005204.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Le retard baisse de 2018 (7,1 minutes) a 2021 (4,7 minutes), puis remonte
regulierement jusqu'a 7,3 minutes en 2025. Le creux de 2020 et 2021 correspond a
la periode ou le trafic etait fortement reduit : moins de trains en circulation,
donc moins de retards en cascade.

C'est une information a garder en tete : ces deux annees ne sont pas
representatives du fonctionnement normal du reseau.

In [15]:
# On regroupe par saison
# .reindex() impose l'ordre des saisons, sinon pandas les range par ordre alphabetique
retard_par_saison = df.groupby("season")["avg_delay_all_arr"].mean()
retard_par_saison = retard_par_saison.reindex(["Hiver", "Printemps", "Ete", "Automne"])

plt.bar(retard_par_saison.index, retard_par_saison.values, color=COULEUR)

plt.title("Retard moyen par saison")
plt.xlabel("Saison")
plt.ylabel("Retard moyen (minutes)")
plt.show()

/tmp/ipykernel_5920/1043795721.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Ete (7,2 minutes) et automne (6,6 minutes) sont les saisons les plus degradees,
le printemps est la plus fiable (5,3 minutes). L'ecart entre la meilleure et la
pire saison est de pres de 2 minutes, soit environ un tiers du retard moyen.

Remarque sur le sujet : il demande de comparer les retards selon l'heure de la
journee. Ce n'est pas possible ici, le dataset est mensuel et ne contient ni
horaire ni jour precis. Les variables temporelles disponibles sont le mois,
le trimestre, la saison et l'annee.

## 6. Autres comparaisons

In [16]:
# Retard moyen selon le type de service, national ou international
retard_par_service = df.groupby("service")["avg_delay_all_arr"].mean()

plt.bar(retard_par_service.index, retard_par_service.values, color=COULEUR)

plt.title("Retard moyen selon le type de service")
plt.xlabel("Service")
plt.ylabel("Retard moyen (minutes)")
plt.show()

/tmp/ipykernel_5920/2898608294.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Retard moyen selon la duree du trajet (Court, Moyen, Long)
retard_par_duree = df.groupby("journey_length", observed=True)[
    "avg_delay_all_arr"
].mean()
retard_par_duree = retard_par_duree.reindex(["Court", "Moyen", "Long"])

plt.bar(retard_par_duree.index, retard_par_duree.values, color=COULEUR)

plt.title("Retard moyen selon la duree du trajet")
plt.xlabel("Duree du trajet")
plt.ylabel("Retard moyen (minutes)")
plt.show()

/tmp/ipykernel_5920/2733072126.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Les trains internationaux accusent 8,1 minutes de retard moyen contre 6,0 pour
les trains nationaux, soit 36 % de plus.

L'effet de la duree est encore plus marque : 3,9 minutes pour les trajets courts
(moins de 1h30), 7,8 minutes pour les longs (plus de 3h), soit le double.
Plus le trajet est long, plus il y a d'occasions de perdre du temps.

In [18]:
# plt.scatter() trace un nuage de points : chaque point est un trajet
# x = duree du trajet, y = retard moyen
# alpha=0.3 rend les points transparents, ce qui montre les zones de forte densite
# s=10 fixe la taille des points
plt.scatter(df["journey_time"], df["avg_delay_all_arr"], alpha=0.3, s=10, color=COULEUR)

plt.title("Retard moyen en fonction de la duree du trajet")
plt.xlabel("Duree du trajet (minutes)")
plt.ylabel("Retard moyen a l'arrivee (minutes)")
plt.show()

/tmp/ipykernel_5920/2307830749.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Le nuage confirme le lien : il s'etale vers le haut quand la duree augmente.
La relation existe mais elle est diffuse, la duree seule ne suffit pas a predire
le retard.

## 7. Causes des retards

In [19]:
# Liste des colonnes qui donnent le pourcentage de retard attribue a chaque cause
colonnes_causes = [
    "pct_cause_external",
    "pct_cause_infra",
    "pct_cause_traffic",
    "pct_cause_rolling_stock",
    "pct_cause_station",
    "pct_cause_passenger",
]

# .mean() calcule la moyenne de chaque colonne
# .sort_values() trie du plus petit au plus grand pour un graphique lisible
causes_moyennes = df[colonnes_causes].mean().sort_values()

causes_moyennes.plot(kind="barh", color=COULEUR)

plt.title("Part moyenne de chaque cause de retard")
plt.xlabel("Part moyenne (%)")
plt.ylabel("Cause")
plt.show()

/tmp/ipykernel_5920/1069030847.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Quatre causes se partagent l'essentiel des retards, avec des parts proches :
infrastructure (22 %), causes externes (22 %), gestion du trafic (20 %) et
materiel roulant (19 %). La gestion en gare et l'affluence voyageurs pesent
beaucoup moins, environ 7 % chacune.

Aucune cause ne domine, ce qui explique pourquoi le retard est difficile a
predire : il resulte de facteurs multiples et independants.

## 8. Correlations

In [20]:
# On choisit les variables numeriques les plus interessantes
colonnes_corr = [
    "avg_delay_all_arr",
    "avg_delay_all_dep",
    "avg_delay_late_arr",
    "delay_rate_arr",
    "punctuality_rate",
    "cancellation_rate",
    "journey_time",
    "nb_scheduled",
    "nb_delayed_15",
    "nb_delayed_60",
]

# .corr() calcule la correlation entre chaque paire de colonnes
# Le resultat va de -1 (les deux variables evoluent en sens inverse)
# a +1 (elles evoluent ensemble), 0 signifiant aucun lien
matrice = df[colonnes_corr].corr()

# On agrandit la figure car la matrice contient beaucoup de cases
plt.figure(figsize=(10, 8))

# sns.heatmap() colorie chaque case selon sa valeur
# annot=True ecrit la valeur dans la case, fmt=".2f" l'affiche avec 2 decimales
# cmap="coolwarm" : bleu pour les correlations negatives, rouge pour les positives
# center=0 : le blanc correspond a une correlation nulle
sns.heatmap(matrice, annot=True, fmt=".2f", cmap="coolwarm", center=0)

plt.title("Correlations entre les variables numeriques")
plt.show()

/tmp/ipykernel_5920/427624621.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# On regarde uniquement la colonne de la variable a predire
# .sort_values(ascending=False) classe de la correlation la plus forte a la plus faible
correlations_cible = matrice["avg_delay_all_arr"].sort_values(ascending=False)
print(correlations_cible.round(2))

avg_delay_all_arr     1.00
delay_rate_arr        0.68
avg_delay_late_arr    0.47
nb_delayed_15         0.44
nb_delayed_60         0.43
journey_time          0.33
avg_delay_all_dep     0.30
cancellation_rate     0.03
nb_scheduled         -0.13
punctuality_rate     -0.68
Name: avg_delay_all_arr, dtype: float64


Lecture des correlations avec le retard a l'arrivee :

- `delay_rate_arr` (+0,68) et `punctuality_rate` (-0,68) sont les plus liees, ce qui
  est logique : plus la part de trains en retard est elevee, plus le retard moyen
  monte. Attention, ces deux variables sont calculees a partir des memes trains,
  elles risquent de faire fuiter l'information dans le modele
- `journey_time` (+0,33) confirme l'effet de la duree vu plus haut
- `avg_delay_all_dep` (+0,30) : un train qui part en retard arrive en retard, mais
  le lien est loin d'etre total, une partie du retard se cree pendant le trajet
- `nb_scheduled` (-0,13) : les liaisons a fort trafic sont plutot mieux tenues

Si on met de cote ces variables derivees du retard, la correlation la plus forte
tombe a 0,33. Il n'existe donc pas de predicteur unique et evident, un modele
multivariable est necessaire.

## 9. Synthese

Ce qu'il faut retenir pour le Step 3 :

1. Le retard moyen a l'arrivee vaut 6,2 minutes, avec une distribution etiree
   vers la droite et quelques valeurs extremes.
2. La gare de depart discrimine fortement : de 2,8 a 14,4 minutes de retard moyen
   selon la gare. A encoder comme variable categorielle.
3. Le mois et la saison ont un effet clair, avec un pic en juillet. A garder.
4. La duree du trajet double le retard entre trajets courts et longs. A garder.
5. Le service international ajoute environ 2 minutes. A garder.
6. Aucune variable ne suffit seule : hors variables derivees du retard, la
   correlation maximale est de 0,33.
7. Les variables `delay_rate_arr`, `punctuality_rate` et les `nb_delayed_*` sont
   calculees a partir du retard lui-meme. Les utiliser comme predicteurs
   reviendrait a donner la reponse au modele, il faudra les ecarter.